In [1]:
%pip install anthropic

You should consider upgrading via the '/Users/nirashanelki/Documents/Works/Research/codes/factual_inconsistency_validate_xai/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import anthropic
import json
import csv
import time
from pathlib import Path

client = anthropic.Anthropic()

In [2]:
def prepare_batch_requests(csv_file_path, output_jsonl_path, start_index=0, end_index=100):
    """
    Convert CSV data into batch API JSONL format.
    Each line is a separate request for KG triplet extraction.
    
    Args:
        csv_file_path: Path to input CSV file
        output_jsonl_path: Path to output JSONL file
        start_index: Starting row index (0-based, inclusive)
        end_index: Ending row index (0-based, exclusive)
    """
    with open(csv_file_path, 'r', encoding='utf-8') as csv_file, \
         open(output_jsonl_path, 'w', encoding='utf-8') as jsonl_file:
        
        reader = csv.DictReader(csv_file)
        
        for idx, row in enumerate(reader):
            # Skip rows before start_index
            if idx < start_index:
                continue
            # Stop after end_index
            if idx >= end_index:
                break
                
            text_id = row['id']  # Adjust column name as needed
            text_content = row['summary']  # Adjust column name as needed
            
            # Create the batch request
            batch_request = {
                "custom_id": f"kg-extraction-{idx}",
                "params": {
                    "model": "claude-sonnet-4-20250514",
                    # "model": "claude-sonnet-4-5-20250929",
                    "system": "You are a helpful assistant in knowledge graph extraction.",
                    "max_tokens": 20000,
                    "messages": [
                        {
                            "role": "user",
                            "content": f"""
        Extract a knowledge graph (KG) from the following text. Follow these steps:
        1. **Entities**: Identify all entities in the text. Ensure each entity is precise and specific.
        2. **Relations**: Extract relationships between entities as triples: ["entity1", "relation", "entity2"].
        3. **Coreference Resolution**: Unify references to the same entity (e.g., "Apple Inc." and "Apple" should be the same
        entity).

        **Important Requirements**:
        - The KG must not be empty. Ensure at least one triple is extracted.
        - All entities mentioned in the text must be included in the KG, either as part of a triple or as a standalone entity if no relation is found.
        - If no explicit relation is found between entities, create a generic relation like "related to" or "associated with" to ensure all entities are connected.
        - Each triple must have three non-empty elements: ["entity1", "relation", "entity2"]. None of these elements can be empty or null.
        - Include temporal or locational context as separate entities (e.g., "in 2020", "at New York").
        
        Please only return the KG as a Python list of triples. For example:
        <python>
        [
        ["Apple Inc.", "founded by", "Steve Jobs"],
        ["Apple Inc.", "headquartered in", "Cupertino, California"],
        ["Apple Inc.", "produces", "iPhone"],
        ["Steve Jobs", "associated with", "Cupertino, California"]
        ]
        </python>
        Text: {text_content}
    """
                        }
                    ]
                }
            }
            
            jsonl_file.write(json.dumps(batch_request) + '\n')
    
    print(f"Batch requests prepared and saved to {output_jsonl_path}")

In [4]:
def submit_batch(jsonl_file_path):
    """
    Submit the batch of requests to Claude API.
    Returns the batch ID for tracking.
    """
    # Read all requests from the JSONL file
    requests = []
    with open(jsonl_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            requests.append(json.loads(line))
    
    # Create the message batch
    message_batch = client.messages.batches.create(
        requests=requests
    )
    
    print(f"Batch submitted successfully!")
    print(f"Batch ID: {message_batch.id}")
    print(f"Status: {message_batch.processing_status}")
    
    return message_batch.id

def check_batch_status(batch_id):
    """
    Check the status of a batch processing job.
    """
    batch = client.messages.batches.retrieve(batch_id)
    
    print(f"\nBatch Status: {batch.processing_status}")
    print(f"Requests: {batch.request_counts}")
    
    return batch.processing_status

def retrieve_batch_results(batch_id, output_jsonl_path):
    """
    Retrieve results once batch processing is complete.
    """
    # Get all results
    results = client.messages.batches.results(batch_id)
    
    with open(output_jsonl_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result.model_dump()) + '\n')
    
    print(f"Results saved to {output_jsonl_path}")

def process_results(results_jsonl_path, output_csv_path):
    """
    Process the batch results and extract KG triplets into a CSV.
    """
    triplets = []
    
    with open(results_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            result = json.loads(line)
            
            # Extract custom_id to get original text ID
            custom_id = result['custom_id']
            text_id = custom_id.replace('kg-extraction-', '')
            
            # Check if request was successful
            if result['result']['type'] == 'succeeded':
                response = result['result']['message']
                
                # Extract the triplets from the response
                content = response['content'][0]['text']
                
                try:
                    # Parse the JSON response
                    extracted_triplets = json.loads(content)
                    
                    # Add text_id to each triplet
                    for triplet in extracted_triplets:
                        triplets.append({
                            'text_id': text_id,
                            'subject': triplet['subject'],
                            'predicate': triplet['predicate'],
                            'object': triplet['object']
                        })
                except json.JSONDecodeError:
                    print(f"Warning: Could not parse response for {custom_id}")
            else:
                print(f"Error processing {custom_id}: {result['result']}")
    
    # Write to CSV
    with open(output_csv_path, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['text_id', 'subject', 'predicate', 'object']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        writer.writeheader()
        writer.writerows(triplets)
    
    print(f"Processed {len(triplets)} triplets and saved to {output_csv_path}")


In [6]:
def main():
    """
    Main workflow for batch KG extraction.
    """
    #     # File paths
    input_csv = "../data/hallucination_test/summcoz_train_sample_100.csv"
    batch_requests_file = "../data/hallucination_test/batch_requests_summaries_1.jsonl"
    batch_results_file = "../data/hallucination_test/batch_results_summaries_1.jsonl"
   
    
    # Step 1: Prepare batch requests
    print("Step 1: Preparing batch requests...")
    prepare_batch_requests(input_csv, batch_requests_file)
    
    # Step 2: Submit batch
    print("\nStep 2: Submitting batch...")
    batch_id = submit_batch(batch_requests_file)
    
    # Step 3: Poll for completion
    print("\nStep 3: Waiting for batch to complete...")
    while True:
        status = check_batch_status(batch_id)
        
        if status == "ended":
            break
        
        print("Still processing... checking again in 60 seconds")
        time.sleep(60)
    
    # Step 4: Retrieve results
    print("\nStep 4: Retrieving results...")
    retrieve_batch_results(batch_id, batch_results_file)
    
    # # Step 5: Process results
    # print("\nStep 5: Processing results into CSV...")
    # process_results(batch_results_file, output_csv)
    
    # print("\nComplete! Your KG triplets are in", output_csv)


In [7]:
if __name__ == "__main__":
    main()

Step 1: Preparing batch requests...
Batch requests prepared and saved to ../data/hallucination_test/batch_requests_summaries_1.jsonl

Step 2: Submitting batch...
Batch submitted successfully!
Batch ID: msgbatch_01Qv8QjnxNMrf1ZAA18XyAyz
Status: in_progress

Step 3: Waiting for batch to complete...

Batch Status: in_progress
Requests: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=100, succeeded=0)
Still processing... checking again in 60 seconds

Batch Status: in_progress
Requests: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=100, succeeded=0)
Still processing... checking again in 60 seconds

Batch Status: in_progress
Requests: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=100, succeeded=0)
Still processing... checking again in 60 seconds

Batch Status: ended
Requests: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=100)

Step 4: Retrieving results...
Results saved to ../data/h